# 05 - Player Similarity

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import pandas as pd

df = pd.read_csv(ROOT / "data/processed/midfielder_clusters_v2.csv")


for cluster in sorted(df["cluster"].unique()):
    print(f"CLUSTER: {cluster}")


    players = df[df["cluster"] == cluster]

    display(
        players[
            [
                "player_name",
                "minutes_played",
                "scoring.expected_goals_per90",
                "scoring.expected_assists_per90",
                "defending.tackles_per90",
                "defending.interceptions_per90",
                "passing.key_passes_per90",
                "passing.accurate_crosses_per90"
            ]
        ].sort_values(
            "scoring.expected_assists_per90",
            ascending=False
        ).head(10)
    )


CLUSTER: 0


,player_name,minutes_played,scoring.expected_goals_per90,scoring.expected_assists_per90,defending.tackles_per90,defending.interceptions_per90,passing.key_passes_per90,passing.accurate_crosses_per90
58,Rayan Cherki,1794,0.197157,0.412876,1.304348,0.351171,3.060201,1.204013
60,Bruno Fernandes,3066,0.303229,0.378082,1.585127,0.587084,4.021526,1.614481
31,Adam Wharton,2562,0.051639,0.250117,2.459016,0.948478,1.545667,0.983607
17,Mathias Jensen,2277,0.085375,0.229644,1.620553,0.830040,1.383399,1.501976
3,Martin Ødegaard,1370,0.078175,0.229270,1.248175,0.459854,2.627737,0.394161
18,Mikkel Damsgaard,2078,0.108277,0.226083,2.468720,1.082772,1.819057,1.212705
22,Pascal Groß,1636,0.076467,0.218949,1.595355,0.660147,2.145477,2.200489
57,Phil Foden,2107,0.251590,0.194352,1.452302,0.384433,2.349312,1.708590
1,Declan Rice,3099,0.088577,0.194288,2.032914,1.074540,1.858664,1.364956
86,Xavi Simons,1769,0.149576,0.187224,1.729791,0.763143,1.780667,0.712267


CLUSTER: 1


,player_name,minutes_played,scoring.expected_goals_per90,scoring.expected_assists_per90,defending.tackles_per90,defending.interceptions_per90,passing.key_passes_per90,passing.accurate_crosses_per90
32,Daichi Kamada,1908,0.075472,0.148113,2.971698,0.660377,0.943396,0.518868
64,Mason Mount,1018,0.127308,0.137917,2.121807,0.795678,1.237721,0.088409
19,Vitaly Janelt,1449,0.054037,0.136646,1.677019,1.863354,1.304348,1.180124
70,Lewis Miley,1494,0.076506,0.121687,1.746988,1.204819,0.903614,0.361446
77,Granit Xhaka,2903,0.030382,0.111299,1.550121,0.899070,1.054082,0.713055
28,Jordan Henderson,1929,0.014930,0.110109,1.259720,0.793157,0.886470,0.793157
34,Will Hughes,1577,0.059924,0.103868,1.540900,0.856056,1.198478,0.684845
85,Sandro Tonali,2543,0.059103,0.097326,1.238694,1.097129,1.132521,0.707825
46,Ao Tanaka,1326,0.103846,0.095701,1.832579,1.425339,1.221719,0.950226
68,Joe Willock,1011,0.108605,0.093472,1.780415,0.801187,1.246291,0.267062


CLUSTER: 2


,player_name,minutes_played,scoring.expected_goals_per90,scoring.expected_assists_per90,defending.tackles_per90,defending.interceptions_per90,passing.key_passes_per90,passing.accurate_crosses_per90
12,David Brooks,1220,0.449262,0.235328,1.401639,0.295082,1.918033,0.811475
56,Enzo Fernández,3121,0.311439,0.199840,1.528356,0.605575,1.989747,0.461391
53,Florian Wirtz,2388,0.256658,0.186935,1.281407,0.414573,2.298995,0.188442
38,Kiernan Dewsbury-Hall,2635,0.142087,0.176926,1.776091,0.409867,1.263757,0.717268
13,Justin Kluivert,951,0.211041,0.151420,0.946372,0.378549,1.987382,0.757098
66,Jacob Murphy,1627,0.168162,0.146589,0.885065,0.331899,1.272280,1.327597
10,Ross Barkley,865,0.149827,0.135260,1.456647,0.208092,1.040462,0.000000
2,Eberechi Eze,1900,0.229737,0.129789,0.900000,0.521053,0.757895,0.094737
76,Chris Rigg,773,0.094308,0.129237,1.513583,0.465718,1.047865,0.465718
5,Mikel Merino,1035,0.284348,0.122609,2.260870,0.869565,1.043478,0.000000


In [3]:
import joblib
import numpy as np
import pandas as pd

df = pd.read_csv(
    ROOT / "data/processed/midfielder_clusters_v2.csv"
)

model_bundle = joblib.load(
    ROOT / "models/kmeans_v2.joblib"
)

scaler = model_bundle["scaler"]

features = [
    "scoring.expected_goals_per90",
    "scoring.expected_assists_per90",
    "defending.tackles_per90",
    "defending.interceptions_per90",
    "passing.key_passes_per90",
    "passing.accurate_crosses_per90",
]

# Preserve the feature order used during training, when available.
if hasattr(scaler, "feature_names_in_"):
    features = list(scaler.feature_names_in_)

# Use the saved scaler; do not fit it again.
X_scaled = scaler.transform(df[features])

In [4]:
player_name = "Florian Wirtz"

matches = np.flatnonzero(
    df["player_name"].eq(player_name).to_numpy()
)

if len(matches) != 1:
    raise ValueError(
        f"Expected one match for {player_name!r}, found {len(matches)}."
    )

player_position = matches[0]

# Euclidean distance across the standardized features.
distances = np.linalg.norm(
    X_scaled - X_scaled[player_position],
    axis=1,
)

results = df.copy()
results["distance"] = distances

# Exclude the selected player.
results = results.iloc[
    np.arange(len(results)) != player_position
]

display(
    results[
        ["player_name", "cluster", "minutes_played", "distance", *features]
    ]
    .sort_values("distance")
    .head(5)
    .round(3)
)

,player_name,cluster,minutes_played,distance,scoring.expected_goals_per90,scoring.expected_assists_per90,defending.tackles_per90,defending.interceptions_per90,passing.key_passes_per90,passing.accurate_crosses_per90
56,Enzo Fernández,2,3121,1.081,0.311,0.200,1.528,0.606,1.990,0.461
13,Justin Kluivert,2,951,1.429,0.211,0.151,0.946,0.379,1.987,0.757
42,Oscar Bobb,0,1050,1.964,0.123,0.173,1.800,0.771,1.886,0.343
86,Xavi Simons,0,1769,1.986,0.150,0.187,1.730,0.763,1.781,0.712
30,Morgan Rogers,2,3285,2.058,0.179,0.113,1.151,0.356,1.288,0.082


In [5]:
from src.similarity import find_similar

similar_players = find_similar(
    player_name="Ryan Gravenberch",
    df=df,
    scaler=scaler,
    features=features,
    top_n=5
)

similar_players



,player_id,season_id,competition_id,competition_name,team_id,position,rating,appearances,starts,minutes_played,...,discipline.penalty_won,discipline.penalty_conceded,scoring.expected_goals_per90,scoring.expected_assists_per90,defending.tackles_per90,defending.interceptions_per90,passing.key_passes_per90,passing.accurate_crosses_per90,cluster,distance
25,pl_96184272,sn_6125938,comp_3039,Premier League,tm_6573,M,6.69,18,12,1519,...,0,0,0.051547,0.077024,1.836735,1.303489,0.770244,0.059250,1,0.381095
69,pl_84259945,sn_6125938,comp_3039,Premier League,tm_7938,M,6.99,27,17,1953,...,0,0,0.106452,0.059908,1.981567,1.382488,0.783410,0.092166,1,0.708497
33,pl_6582836,sn_6125938,comp_3039,Premier League,tm_3498,M,6.32,31,28,1730,...,0,0,0.090520,0.074393,1.508671,1.300578,0.728324,0.260116,1,0.818644
61,pl_58883966,sn_6125938,comp_3039,Premier League,tm_1552,M,6.70,31,11,1677,...,0,0,0.038640,0.038640,1.985689,1.556351,0.483005,0.000000,1,0.844579
4,pl_30477667,sn_6125938,comp_3039,Premier League,tm_9145,M,7.13,38,25,3001,...,0,0,0.081273,0.065078,2.039320,1.169610,0.569810,0.089970,1,0.906983


In [6]:
similar_players = find_similar(
    player_name="Dominik Szoboszlai",
    df=df,
    scaler=scaler,
    features=features,
    top_n=5
)

similar_players

,player_id,season_id,competition_id,competition_name,team_id,position,rating,appearances,starts,minutes_played,...,discipline.penalty_won,discipline.penalty_conceded,scoring.expected_goals_per90,scoring.expected_assists_per90,defending.tackles_per90,defending.interceptions_per90,passing.key_passes_per90,passing.accurate_crosses_per90,cluster,distance
22,pl_185007,sn_6125938,comp_3039,Premier League,tm_1552,M,7.39,19,17,1636,...,0,0,0.076467,0.218949,1.595355,0.660147,2.145477,2.200489,0,1.046345
45,pl_29837726,sn_6125938,comp_3039,Premier League,tm_0256,M,6.77,29,20,2378,...,0,0,0.105593,0.163877,2.460050,0.756939,2.384357,1.703112,0,1.373197
1,pl_08343894,sn_6125938,comp_3039,Premier League,tm_9145,M,7.70,36,30,3099,...,0,0,0.088577,0.194288,2.032914,1.074540,1.858664,1.364956,0,1.454524
17,pl_29661058,sn_6125938,comp_3039,Premier League,tm_1662,M,6.66,36,23,2277,...,0,0,0.085375,0.229644,1.620553,0.830040,1.383399,1.501976,0,1.628423
57,pl_45889667,sn_6125938,comp_3039,Premier League,tm_3039,M,7.26,33,23,2107,...,0,0,0.251590,0.194352,1.452302,0.384433,2.349312,1.708590,0,1.756984


In [ ]:
similar_players = find_similar(
    player_name="Alexis Mac Allister",
    df=df,
    scaler=scaler,
    features=features,
    top_n=5
)

similar_players

,player_id,season_id,competition_id,competition_name,team_id,position,rating,appearances,starts,minutes_played,...,discipline.penalty_won,discipline.penalty_conceded,scoring.expected_goals_per90,scoring.expected_assists_per90,defending.tackles_per90,defending.interceptions_per90,passing.key_passes_per90,passing.accurate_crosses_per90,cluster,distance
68,pl_13745878,sn_6125938,comp_3039,Premier League,tm_7938,M,6.68,24,15,1011,...,0,0,0.108605,0.093472,1.780415,0.801187,1.246291,0.267062,1,0.611770
35,pl_96970282,sn_6125938,comp_3039,Premier League,tm_8091,M,5.32,20,14,730,...,0,0,0.145479,0.064110,1.726027,0.369863,1.232877,0.369863,2,0.798684
90,pl_57489228,sn_6125938,comp_3039,Premier League,tm_5104,M,6.55,26,13,1251,...,0,0,0.067626,0.076978,2.014388,0.719424,1.007194,0.359712,1,0.848528
36,pl_15464654,sn_6125938,comp_3039,Premier League,tm_8091,M,5.35,13,8,537,...,0,0,0.125698,0.033520,2.178771,0.670391,0.670391,0.000000,1,1.168946
11,pl_62854502,sn_6125938,comp_3039,Premier League,tm_1772,M,7.24,37,22,2859,...,0,0,0.115845,0.090031,1.888772,1.133263,0.881427,0.440714,1,1.211603
